In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_NSIT Dwarka, Delhi - CPCB.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,...,Benzene,Toluene,Eth-Benzene,MP-Xylene,RH,WS,WD,BP,Xylene,AT
0,01-01-2025 00:00,02-01-2025 00:00,88.87,199.83,23.57,41.60,41.28,42.82,13.47,1.12,...,2.61,4.37,2.31,2.22,88.35,0.35,280.39,NaN,NaN,NaN
1,02-01-2025 00:00,03-01-2025 00:00,90.03,214.54,23.63,40.42,40.71,41.94,12.46,0.95,...,2.62,4.38,2.31,2.23,90.43,0.44,276.82,NaN,NaN,NaN
2,03-01-2025 00:00,04-01-2025 00:00,118.84,283.92,25.51,53.49,49.19,53.56,12.98,0.90,...,2.61,4.36,2.32,2.21,89.41,0.24,191.94,NaN,NaN,NaN
3,04-01-2025 00:00,05-01-2025 00:00,102.74,230.93,23.71,48.70,45.17,46.89,14.89,1.33,...,2.61,4.40,2.31,2.21,86.26,0.48,148.59,NaN,NaN,NaN
4,05-01-2025 00:00,06-01-2025 00:00,89.82,193.93,23.26,42.64,41.58,42.01,12.83,1.10,...,2.58,4.35,2.30,2.19,85.92,0.84,157.99,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,107.62,269.28,19.96,75.26,55.61,33.37,7.99,0.72,...,3.30,4.04,2.81,2.82,58.60,0.19,221.35,NaN,NaN,NaN
316,13-11-2025 00:00,14-11-2025 00:00,119.84,297.47,10.81,75.83,48.77,45.90,8.02,0.72,...,3.29,4.06,2.80,2.83,64.30,0.14,219.85,NaN,NaN,NaN
317,14-11-2025 00:00,15-11-2025 00:00,98.75,474.92,8.04,73.21,44.93,43.48,8.88,0.71,...,3.28,4.06,2.78,2.83,59.41,0.18,213.94,NaN,NaN,NaN
318,15-11-2025 00:00,16-11-2025 00:00,99.14,271.78,14.07,76.28,51.68,39.08,11.15,0.78,...,3.28,4.06,2.79,2.83,62.89,0.17,213.57,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['BP', 'Xylene']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date      0
To Date        0
PM2.5          0
PM10           0
NO             0
NO2            0
NOx            0
NH3            0
SO2            0
CO             0
Ozone          0
Benzene        0
Toluene        0
Eth-Benzene    0
MP-Xylene      0
RH             0
WS             0
WD             0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 18)
          From Date           To Date   PM2.5    PM10    NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   88.87  199.83  9.39  41.60  41.28   
1  02-01-2025 00:00  03-01-2025 00:00   90.03  214.54  9.39  40.42  40.71   
2  03-01-2025 00:00  04-01-2025 00:00  118.84  283.92  9.39  53.49  49.19   
3  04-01-2025 00:00  05-01-2025 00:00  102.74  230.93  9.39  48.70  45.17   
4  05-01-2025 00:00  06-01-2025 00:00   89.82  193.93  9.39  42.64  41.58   

     NH3    SO2    CO  Ozone  Benzene  Toluene  Eth-Benzene  MP-Xylene     RH  \
0  42.82  13.47  1.12  15.16     2.61     4.37         2.31       2.22  88.35   
1  41.94  12.46  0.95  15.74     2.62     4.38         2.31       2.23  90.43   
2  53.56  12.98  0.90  15.59     2.61     4.36         2.32       2.21  89.41   
3  46.89  14.89  1.33  18.71     2.61     4.40         2.31       2.21  86.26   
4  42.01  12.83  1.10  15.93     2.58     4.35         2.30       2.19  85.92   

     WS      WD  
0  0.35  

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,Eth-Benzene,MP-Xylene,RH,WS,WD
0,01-01-2025 00:00,02-01-2025 00:00,0.755622,0.763843,0.233979,0.040937,1.087315,0.331405,0.647710,-0.085421,-0.603297,1.394347,1.257269,1.442790,0.871400,1.293159,-0.552998,1.596863
1,02-01-2025 00:00,03-01-2025 00:00,0.789602,0.954113,0.233979,-0.053827,1.026690,0.252956,0.391235,-0.376032,-0.541674,1.427403,1.267977,1.442790,0.895055,1.397076,-0.171711,1.528506
2,03-01-2025 00:00,04-01-2025 00:00,1.633548,1.851522,0.233979,0.995807,1.928612,1.288843,0.523282,-0.461506,-0.557611,1.394347,1.246561,1.470298,0.847744,1.346117,-1.019015,-0.096741
3,04-01-2025 00:00,05-01-2025 00:00,1.161923,1.166112,0.233979,0.611129,1.501050,0.694233,1.008299,0.273570,-0.226127,1.394347,1.289394,1.442790,0.847744,1.188742,-0.002251,-0.926789
4,05-01-2025 00:00,06-01-2025 00:00,0.783450,0.687528,0.233979,0.124458,1.119222,0.259196,0.485192,-0.119610,-0.521488,1.295178,1.235853,1.415282,0.800434,1.171755,1.522895,-0.746802
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,1.304875,1.662158,0.233979,0.004798,2.611434,-0.511033,-0.743857,-0.769212,1.098746,-0.093177,0.903901,2.818185,2.290714,-0.193162,-1.230841,0.466389
316,13-11-2025 00:00,14-11-2025 00:00,1.662842,2.026787,0.666456,0.004798,1.883941,0.605978,-0.736239,-0.769212,1.413231,-0.093177,0.925317,2.790677,2.314369,0.091612,-1.442666,0.437668
317,14-11-2025 00:00,15-11-2025 00:00,1.045042,-0.052336,-0.177179,2.579494,1.475524,0.390242,-0.517854,-0.786307,1.218803,-0.093177,0.925317,2.735662,2.314369,-0.152695,-1.273206,0.324506
318,15-11-2025 00:00,16-11-2025 00:00,1.056466,1.694495,1.659326,0.004798,2.193445,-0.002004,0.058580,-0.666643,1.279362,-0.093177,0.925317,2.763169,2.314369,0.021168,-1.315571,0.317421


In [10]:
df.to_excel('NSITDwarka2025.xlsx', index=False)